In [2]:
!pip install "bottleneck>=1.4.2" "numexpr>=2.10.2" "numpy<2" "pandas" "openpyxl" "scikit-learn" --upgrade

  Obtaining dependency information for bottleneck>=1.4.2 from https://files.pythonhosted.org/packages/6f/42/01d4920b0aa51fba503f112c90714547609bbe17b6ecfc1c7ae1da3183df/bottleneck-1.6.0-cp311-cp311-win_amd64.whl.metadata
   ---------------------------------------- 0.0/113.4 kB ? eta -:--:--
   ---------------------------------------- 113.4/113.4 kB 2.2 MB/s eta 0:00:00
  Attempting uninstall: bottleneck
    Found existing installation: Bottleneck 1.3.5
    Uninstalling Bottleneck-1.3.5:
      Successfully uninstalled Bottleneck-1.3.5


In [30]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import itertools
import datetime as dt

In [2]:
# Set up relative path to data directory
file_path = '../data/Online Retail.xlsx'

In [3]:
# Load raw Excel data using openpyxl engine
df_raw = pd.read_excel(file_path)
print(f"Dataset successfully loaded! Shape: {df_raw.shape}")

Dataset successfully loaded! Shape: (541909, 8)


In [5]:
print(f"Initial Dataset Shape: {df_raw.shape}")
df_raw.head()

Initial Dataset Shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


# Data Cleaning Pipeline

In [6]:
# Drop missing CustomerIDs
df_clean = df_raw.dropna(subset=['CustomerID']).copy()

In [7]:
# Convert CustomerID to integer format
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

In [8]:
# Drop duplicate records
df_clean = df_clean.drop_duplicates()

In [9]:
# Remove cancelled transactions (InvoiceNo starting with 'C') and invalid quantities/prices
df_clean['InvoiceNo'] = df_clean['InvoiceNo'].astype(str)
df_clean = df_clean[~df_clean['InvoiceNo'].str.startswith('C')]
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]

In [10]:
# Clean up string fields
df_clean['Description'] = df_clean['Description'].astype(str).str.strip()

In [11]:
print(f"Cleaned Dataset Shape: {df_clean.shape}")
print(f"Unique Customers: {df_clean['CustomerID'].nunique()}")

Cleaned Dataset Shape: (392692, 8)
Unique Customers: 4338


# Feature Engineering - Transaction Metrics

Calculate total order value per line item and extract datetime attributes for time-based analysis.

In [12]:
# Calculate Total Spending per line item
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']

In [13]:
# Date feature extraction
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['TransactionDate'] = df_clean['InvoiceDate'].dt.date
df_clean['YearMonth'] = df_clean['InvoiceDate'].dt.to_period('M')
df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.day_name()
df_clean['Hour'] = df_clean['InvoiceDate'].dt.hour

In [14]:
df_clean.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount,TransactionDate,YearMonth,DayOfWeek,Hour
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,2010-12-01,2010-12,Wednesday,8
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12-01,2010-12,Wednesday,8
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,2010-12-01,2010-12,Wednesday,8
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12-01,2010-12,Wednesday,8
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12-01,2010-12,Wednesday,8


# Building the RFM (Recency, Frequency, Monetary) Table

Construct customer-level features for K-Means Clustering.

In [17]:
# Set snapshot reference date (1 day after max date in dataset)
snapshot_date = df_clean['InvoiceDate'].max() + dt.timedelta(days=1)

In [18]:
# Aggregate RFM features per customer
rfm = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',                                  # Frequency
    'TotalAmount': 'sum'                                      # Monetary
}).reset_index()

In [19]:
# Rename columns
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

In [20]:
print(f"RFM Table Shape: {rfm.shape}")
rfm.head()

RFM Table Shape: (4338, 4)


,CustomerID,Recency,Frequency,Monetary
0,12346,326,1,77183.60
1,12347,2,7,4310.00
2,12348,75,4,1797.24
3,12349,19,1,1757.55
4,12350,310,1,334.40


# RFM Scoring & Outlier Handling

Calculate RFM quantile scores and apply logarithmic transformation to handle skewness before machine learning scaling.

In [23]:
# Calculate RFM quantile scores (1 to 4)
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=4, labels=[4, 3, 2, 1]) # Lower recency is better
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=4, labels=[1, 2, 3, 4])

In [24]:
# Combine RFM Segment String and Score
rfm['RFM_Segment'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm['RFM_Score'] = rfm[['R_Score', 'F_Score', 'M_Score']].sum(axis=1)

In [25]:
# Log transformation for ML models to reduce right-skewness
rfm_log = rfm[['Recency', 'Frequency', 'Monetary']].apply(np.log1p)

In [26]:
rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Segment,RFM_Score
0,12346,326,1,77183.60,1,1,4,114,6
1,12347,2,7,4310.00,4,4,4,444,12
2,12348,75,4,1797.24,2,3,4,234,9
3,12349,19,1,1757.55,3,1,4,314,8
4,12350,310,1,334.40,1,1,2,112,4


In [27]:
# Save clean transactions 
df_clean.to_csv('../data/fact_transactions_clean.csv', index=False)

In [28]:
# Save customer-level RFM metrics for ML Clustering
rfm.to_csv('../data/dim_customer_rfm.csv', index=False)

In [29]:
# Save log-transformed RFM features for ML models
rfm_log.to_csv('../data/rfm_log_transformed.csv', index=False)

print("All dataset cleaned and saved successfully!")

All dataset cleaned and saved successfully!


# Executing K-Means Clustering ($K=4$)

In [31]:
# Standardize Log-Transformed Features
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

In [32]:
# Fit K-Means Model (K = 4)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

In [33]:
cluster_labels = {
    0: 'Champions (VIP)',
    1: 'Lost / Hibernating',
    2: 'New / Promising',
    3: 'At Risk'
}

In [34]:
rfm['Customer_Segment'] = rfm['Cluster'].map(cluster_labels)

In [35]:
# Display Segment Distribution Summary
rfm_summary = rfm.groupby('Customer_Segment').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean',
    'CustomerID': 'count'
}).rename(columns={'CustomerID': 'Customer_Count'}).round(2)

In [36]:
print(rfm_summary)

                    Recency  Frequency  Monetary  Customer_Count
Customer_Segment                                                
At Risk               71.64       4.08   1801.78            1166
Champions (VIP)       12.17      13.75   8088.02             713
Lost / Hibernating   181.51       1.32    341.00            1622
New / Promising       17.70       2.19    557.32             837


Map Numeric Clusters to Business Personas based on RFM Averages

Average Metrics per Cluster:

Cluster 0: Recency: ~12 days | Frequency: ~14 orders | Monetary: ~$8,088 (Champions/VIPs)

Cluster 1: Recency: ~182 days | Frequency: ~1 order | Monetary: ~$341 (Lost/Hibernating)

Cluster 2: Recency: ~18 days | Frequency: ~2 orders | Monetary: ~$557 (New/Promising)

Cluster 3: Recency: ~72 days | Frequency: ~4 orders | Monetary: ~$1,802 (At Risk)

In [37]:
# Save output for Power BI integration
rfm.to_csv('../data/dim_customer_rfm_clustered.csv', index=False)

# Market Basket Analysis (FP-Growth)

In [38]:
from itertools import combinations

In [39]:
# Load cleaned transactions
df_clean = pd.read_csv('../data/fact_transactions_clean.csv')

In [40]:
# Filter out non-product or administrative stock codes (optional clean-up)
# Filter for transactions in the UK (largest market) to keep matrix computationally light
df_uk = df_clean[df_clean['Country'] == 'United Kingdom']

In [41]:
# Get unique products per invoice (basket format)
baskets = df_uk.groupby('InvoiceNo')['Description'].apply(lambda x: list(set(x)))

In [42]:
# Filter out single-item orders (need at least 2 items to form a pair)
baskets = baskets[baskets.apply(len) > 1]
total_baskets = len(baskets)

In [43]:
# Count Item Frequencies & Pair Frequencies
item_counts = {}
pair_counts = {}

In [44]:
for basket in baskets:
    # Item counts
    for item in basket:
        item_counts[item] = item_counts.get(item, 0) + 1
    
    # Pair counts (combinations of 2 items)
    for pair in combinations(sorted(basket), 2):
        pair_counts[pair] = pair_counts.get(pair, 0) + 1

In [45]:
# Calculate Support, Confidence, and Lift for pairs
rules_list = []

for (item_A, item_B), pair_count in pair_counts.items():
    support_AB = pair_count / total_baskets
    
    # Filter out rare pairs (min support threshold of 1.5%)
    if support_AB >= 0.015:
        # Direction 1: A -> B
        support_A = item_counts[item_A] / total_baskets
        support_B = item_counts[item_B] / total_baskets
        
        confidence_A_B = pair_count / item_counts[item_A]
        lift_A_B = confidence_A_B / support_B
        
        rules_list.append({
            'Antecedent (Item A)': item_A,
            'Consequent (Item B)': item_B,
            'Support': round(support_AB, 4),
            'Confidence': round(confidence_A_B, 4),
            'Lift': round(lift_A_B, 2)
        })
        
        # Direction 2: B -> A
        confidence_B_A = pair_count / item_counts[item_B]
        lift_B_A = confidence_B_A / support_A
        
        rules_list.append({
            'Antecedent (Item A)': item_B,
            'Consequent (Item B)': item_A,
            'Support': round(support_AB, 4),
            'Confidence': round(confidence_B_A, 4),
            'Lift': round(lift_B_A, 2)
        })

In [46]:
# Convert to DataFrame and sort by Lift
rules_df = pd.DataFrame(rules_list).sort_values(by='Lift', ascending=False).reset_index(drop=True)

In [47]:
# Save rules for Power BI cross-selling Matrix visual
rules_df.to_csv('../data/dim_product_recommendations.csv', index=False)

In [48]:
# Print Top 10 Product Recommendations
rules_df.head(10)

,Antecedent (Item A),Consequent (Item B),Support,Confidence,Lift
0,WOODEN HEART CHRISTMAS SCANDINAVIAN,WOODEN STAR CHRISTMAS SCANDINAVIAN,0.0195,0.6897,26.51
1,WOODEN STAR CHRISTMAS SCANDINAVIAN,WOODEN HEART CHRISTMAS SCANDINAVIAN,0.0195,0.7500,26.51
2,GREEN REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER,0.0263,0.6601,20.75
3,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.0263,0.8262,20.75
4,SPACEBOY LUNCH BOX,DOLLY GIRL LUNCH BOX,0.0204,0.6186,20.36
5,DOLLY GIRL LUNCH BOX,SPACEBOY LUNCH BOX,0.0204,0.6702,20.36
6,JUMBO BAG VINTAGE CHRISTMAS,JUMBO BAG 50'S CHRISTMAS,0.0154,0.6466,18.37
7,JUMBO BAG 50'S CHRISTMAS,JUMBO BAG VINTAGE CHRISTMAS,0.0154,0.4362,18.37
8,ROSES REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER,0.0249,0.5674,17.84
9,PINK REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,0.0249,0.7832,17.84


In [51]:
# Save final clean datasets
df_clean.to_csv('../data/fact_transactions_clean.csv', index=False)
rfm.to_csv('../data/dim_customer_rfm_clustered.csv', index=False)
rules_df.to_csv('../data/dim_product_recommendations.csv', index=False)

print("Data processing complete! All files saved for Power BI import.")

Data processing complete! All files saved for Power BI import.
